In [1]:
import os
import csv
import gc
import json
import time
import logging
import warnings

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

from gluonts.itertools import batcher
from gluonts.model.forecast import QuantileForecast
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gluonts.ev.metrics import (
    MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND,
    MeanWeightedSumQuantileLoss,
)

from gift_eval.data import Dataset
from uni2ts.model.moiraic import MoiraicForecast, MoiraicModule
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

load_dotenv()
logging.getLogger("gluonts.model.forecast").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
print("Imports complete.")

Imports complete.


In [2]:
# Same checkpoint loaded into both Moiraic and Moirai2 (weights are interchangeable).
# MODEL_PATH = "/srv/disk00/ctadler/uni2ts/outputs/pretrain/moiraic/gift_eval_pretrain_weighted/moiraic_training_11/HF_checkpoints/last"
MODEL_PATH = "Salesforce/moirai-2.0-R-small"
CHECKPOINT_TYPE = "hf"   # "hf" or "ckpt"

CONTEXT_LENGTH = 4000
BATCH_SIZE = 128
QUANTILE_LEVELS = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
DEVICE = "cuda:7"

# Datasets — keep small for benchmarking; expand as needed.
SHORT_DATASETS = "ett2/H ett2/D LOOP_SEATTLE/H m4_hourly"
# SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "bizitobs_l2c/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T"
# MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
SHORT_DATASETS = ""
MED_LONG_DATASETS = "ett1/H ett2/H bizitobs_l2c/H jena_weather/H  LOOP_SEATTLE/5T solar/10T electricity/15T"

OUTPUT_DIR = "../results/kv_cache_benchmark"

# Configurations to compare.
CONFIGS = {
    "moiraic_cached": {
        "module_cls": MoiraicModule,
        "forecast_cls": MoiraicForecast,
        "forecast_kwargs": {"use_cache": True, "update_context_in_ar": False},
    },
    # "moiraic_frozen_no_cache": {
    #     "module_cls": MoiraicModule,
    #     "forecast_cls": MoiraicForecast,
    #     "forecast_kwargs": {"use_cache": False, "update_context_in_ar": False},
    # },
    # "moiraic_original_no_cache": {
    #     "module_cls": MoiraicModule,
    #     "forecast_cls": MoiraicForecast,
    #     "forecast_kwargs": {"use_cache": False, "update_context_in_ar": True},
    # },
    "moirai2_baseline": {
        "module_cls": Moirai2Module,
        "forecast_cls": Moirai2Forecast,
        "forecast_kwargs": {},
    },
}

# Pretty name mapping for resolving dataset properties.
PRETTY_NAMES = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}
DATASET_PROPS = json.load(open("/srv/disk00/ctadler/gift-eval/notebooks/dataset_properties.json"))

device = torch.device(DEVICE if DEVICE != "auto"
                      else ("cuda:7" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")

Using device: cuda:7


In [3]:
_module_cache: dict = {}

def load_module(module_cls, model_path: str, checkpoint_type: str = "hf"):
    key = (module_cls.__name__, model_path, checkpoint_type)
    if key in _module_cache:
        return _module_cache[key]
    if checkpoint_type == "hf":
        module = module_cls.from_pretrained(model_path)
    elif checkpoint_type == "ckpt":
        module = module_cls.load_from_checkpoint(model_path)
    else:
        raise ValueError(f"Unknown checkpoint_type: {checkpoint_type!r}")
    _module_cache[key] = module
    print(f"Loaded {module_cls.__name__} from {model_path}")
    return module

def evict_other_modules(active_module_cls):
    # Drop any cached module that's not the one we're about to use.
    keys_to_drop = [k for k in _module_cache if k[0] != active_module_cls.__name__]
    for k in keys_to_drop:
        del _module_cache[k]
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)

class BenchmarkedPredictor:
    """
    Predictor that wraps any (forecast_cls, forecast_kwargs) combination
    and accumulates GPU time / peak VRAM / item count across predict() calls.
    """
    def __init__(
        self,
        module,
        forecast_cls,
        forecast_kwargs,
        prediction_length,
        context_length,
        target_dim,
        past_feat_dynamic_real_dim,
        device,
        batch_size,
        quantile_levels,
    ):
        self.prediction_length = prediction_length
        self.context_length = context_length
        self.device = device
        self.batch_size = batch_size
        self.quantile_levels = quantile_levels

        self.model = forecast_cls(
            module=module,
            prediction_length=prediction_length,
            context_length=context_length,
            target_dim=target_dim,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=past_feat_dynamic_real_dim,
            **forecast_kwargs,
        ).to(self.device)

        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)
            self.model_vram_mb = torch.cuda.memory_allocated(self.device) / (1024**2)
        else:
            self.model_vram_mb = 0.0
        self.reset_stats()
    
    def warmup(self, inputs, n_batches=1):
        """Run a few discarded batches so cuDNN workspaces, etc. are paid for."""
        bs = self.batch_size
        for i, batch in enumerate(batcher(inputs, batch_size=bs)):
            if i >= n_batches:
                break
            past_target = [entry["target"] for entry in batch]
            _ = self.model.predict(past_target)
        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)
        self.reset_stats()  # discard warmup peak

    def reset_stats(self):
        self.total_time_s = 0.0
        self.peak_vram_mb = 0.0
        self.num_items = 0
        if self.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(self.device)

    def predict(self, test_data_input):
        batch_size = self.batch_size
        while True:
            try:
                forecast_quantiles = []

                if self.device.type == "cuda":
                    torch.cuda.synchronize(self.device)
                t0 = time.perf_counter()

                items_in_call = 0
                for batch in batcher(test_data_input, batch_size=batch_size):
                    past_target = [entry["target"] for entry in batch]
                    forecasts = self.model.predict(past_target)
                    forecast_quantiles.append(forecasts)
                    items_in_call += len(past_target)

                if self.device.type == "cuda":
                    torch.cuda.synchronize(self.device)
                self.total_time_s += time.perf_counter() - t0
                self.num_items += items_in_call

                if self.device.type == "cuda":
                    self.peak_vram_mb = max(
                        self.peak_vram_mb,
                        torch.cuda.max_memory_allocated(self.device) / (1024 ** 2),
                    )
                forecast_quantiles = np.concatenate(forecast_quantiles)
                break
            except torch.cuda.OutOfMemoryError:
                batch_size //= 2
                if batch_size < 1:
                    raise
                print(f"OOM — reducing batch_size to {batch_size}")
                gc.collect()
                torch.cuda.empty_cache()

        out = []
        for item, ts in zip(forecast_quantiles, test_data_input):
            forecast_start_date = ts["start"] + len(ts["target"])
            out.append(
                QuantileForecast(
                    item_id=ts["item_id"],
                    forecast_arrays=item,
                    start_date=forecast_start_date,
                    forecast_keys=list(map(str, self.quantile_levels)),
                )
            )
        return out

print("BenchmarkedPredictor defined.")

BenchmarkedPredictor defined.


In [4]:
from itertools import islice


metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=list(QUANTILE_LEVELS)),
]
METRIC_KEYS = [
    "MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]",
    "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]", "ND[0.5]",
    "mean_weighted_sum_quantile_loss",
]
KEEP_METRICS = ["MASE[0.5]", "sMAPE[0.5]", "NRMSE[mean]", "ND[0.5]",
                "mean_weighted_sum_quantile_loss"]

def resolve_ds_config(ds_name, term):
    if "/" in ds_name:
        ds_key, ds_freq = ds_name.split("/", 1)
    else:
        ds_key = ds_name
        ds_freq = DATASET_PROPS[PRETTY_NAMES.get(ds_key.lower(), ds_key.lower())]["frequency"]
    ds_key = PRETTY_NAMES.get(ds_key.lower(), ds_key.lower())
    return ds_key, ds_freq, f"{ds_key}/{ds_freq}/{term}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
results_csv = os.path.join(OUTPUT_DIR, "benchmark.csv")

short_set = set(SHORT_DATASETS.split())
medlong_set = set(MED_LONG_DATASETS.split())
all_datasets = sorted(short_set | medlong_set)

header = ["dataset", "config", "inference_time_s", "peak_vram_mb",
          "throughput_items_per_s", "num_items"] + METRIC_KEYS

if os.path.exists(results_csv):
    os.remove(results_csv)
    print(f"Deleted {results_csv}; re-run the eval cell to regenerate.")

with open(results_csv, "w", newline="") as f:
    csv.writer(f).writerow(header)

for ds_name in all_datasets:
    for term in ["medium", "long"]:
        if term in ("medium", "long") and ds_name not in medlong_set:
            continue

        ds_key, ds_freq, ds_config = resolve_ds_config(ds_name, term)
        print(f"\n{'='*60}\nDataset: {ds_config}\n{'='*60}")

        to_univariate = Dataset(name=ds_name, term=term, to_univariate=False).target_dim != 1
        dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
        season_length = get_seasonality(dataset.freq)

        for config_name, cfg in CONFIGS.items():
            print(f"\n  → {config_name}")
            module = load_module(cfg["module_cls"], MODEL_PATH, CHECKPOINT_TYPE)

            if device.type == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.synchronize(device)

            predictor = BenchmarkedPredictor(
                module=module,
                forecast_cls=cfg["forecast_cls"],
                forecast_kwargs=cfg["forecast_kwargs"],
                prediction_length=dataset.prediction_length,
                context_length=CONTEXT_LENGTH,
                target_dim=1,
                past_feat_dynamic_real_dim=dataset.past_feat_dynamic_real_dim,
                device=device,
                batch_size=BATCH_SIZE,
                quantile_levels=QUANTILE_LEVELS,
            )
            warmup_inputs = list(islice(dataset.test_data.input, predictor.batch_size))
            predictor.warmup(warmup_inputs, n_batches=1)
            predictor.reset_stats()

            res = evaluate_model(
                predictor,
                test_data=dataset.test_data,
                metrics=metrics,
                batch_size=BATCH_SIZE,
                axis=None,
                mask_invalid_label=True,
                allow_nan_forecast=False,
                seasonality=season_length,
            )

            tput = predictor.num_items / predictor.total_time_s if predictor.total_time_s > 0 else float("nan")
            row = [
                ds_config, config_name,
                round(predictor.total_time_s, 4),
                round(predictor.peak_vram_mb, 1),
                round(predictor.model_vram_mb, 1),                                # NEW
                round(predictor.peak_vram_mb - predictor.model_vram_mb, 1),       # NEW: predict-time delta
                round(tput, 3),
                predictor.num_items,
            ] + [float(res[k][0]) for k in METRIC_KEYS]
            with open(results_csv, "a", newline="") as f:
                csv.writer(f).writerow(row)
            print(
                f"    time={predictor.total_time_s:.2f}s | "
                f"peak_vram={predictor.peak_vram_mb:.0f}MB | "
                f"throughput={tput:.2f} items/s | "
                f"MASE={res['MASE[0.5]'][0]:.4f} | "
                f"WQL={res['mean_weighted_sum_quantile_loss'][0]:.4f}"
            )

            # Free predictor between configs.
            del predictor
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

print(f"\n✅ Saved benchmark to {results_csv}")

Deleted ../results/kv_cache_benchmark/benchmark.csv; re-run the eval cell to regenerate.

Dataset: loop_seattle/5T/medium

  → moiraic_cached
Loaded MoiraicModule from Salesforce/moirai-2.0-R-small


6460it [02:04, 51.82it/s]


    time=77.21s | peak_vram=2996MB | throughput=83.67 items/s | MASE=0.8636 | WQL=0.0801

  → moirai2_baseline


Loaded Moirai2Module from Salesforce/moirai-2.0-R-small


6460it [02:01, 52.99it/s]


...

    time=114.59s | peak_vram=9817MB | throughput=13.15 items/s | MASE=0.9995 | WQL=0.4124

Dataset: solar/10T/long

  → moiraic_cached


1096it [00:09, 120.30it/s]


    time=26.45s | peak_vram=3564MB | throughput=41.43 items/s | MASE=1.0094 | WQL=0.4077

  → moirai2_baseline


1096it [00:08, 123.15it/s]


    time=138.81s | peak_vram=10690MB | throughput=7.90 items/s | MASE=1.0108 | WQL=0.4084

✅ Saved benchmark to ../results/kv_cache_benchmark/benchmark.csv


In [10]:
df = pd.read_csv(results_csv, index_col=False)
df.head(10)

/tmp/ipykernel_344657/1961114287.py:1: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(results_csv, index_col=False)


,dataset,config,inference_time_s,peak_vram_mb,throughput_items_per_s,num_items,MSE[mean],MSE[0.5],MAE[0.5],MASE[0.5],MAPE[0.5],sMAPE[0.5],MSIS,RMSE[mean],NRMSE[mean],ND[0.5],mean_weighted_sum_quantile_loss
0,loop_seattle/5T/medium,moiraic_cached,77.2060,2995.5,44.1,2951.5,83.672,6460,91.424300,91.424300,5.439653,0.863598,0.196666,0.122212,10.510166,9.561606,0.170121
1,loop_seattle/5T/medium,moirai2_baseline,492.2377,9816.7,96.3,9720.4,13.124,6460,91.641516,91.641516,5.445967,0.864562,0.196959,0.122339,10.540972,9.572958,0.170323
2,loop_seattle/5T/long,moiraic_cached,116.6479,3564.1,96.3,3467.8,41.535,4845,101.763644,101.763644,5.840416,0.926360,0.202934,0.129540,12.848579,10.087797,0.178354
3,loop_seattle/5T/long,moirai2_baseline,613.9047,10690.3,96.3,10594.0,7.892,4845,102.152833,102.152833,5.852678,0.928242,0.203380,0.129797,12.898493,10.107068,0.178695
4,bizitobs_l2c/H/medium,moiraic_cached,0.1173,260.0,96.3,163.8,59.660,7,73.581297,73.581297,5.040061,0.508572,0.561177,0.795006,10.446439,8.577954,0.519408
5,bizitobs_l2c/H/medium,moirai2_baseline,0.5443,631.8,96.3,535.5,12.860,7,73.605487,73.605487,5.040342,0.508575,0.561661,0.795054,10.486612,8.579364,0.519494
6,bizitobs_l2c/H/long,moiraic_cached,0.2149,291.8,96.3,195.5,32.573,7,100.028044,100.028044,5.769298,0.611829,0.713413,0.807579,14.967456,10.001402,0.610915
7,bizitobs_l2c/H/long,moirai2_baseline,0.9073,681.9,96.3,585.7,7.715,7,99.568868,99.568868,5.755021,0.610783,0.712003,0.807344,15.002673,9.978420,0.609511
8,electricity/15T/medium,moiraic_cached,88.3323,3038.8,96.3,2942.5,83.775,7400,282980.445218,282980.445218,55.662811,0.841188,0.132036,0.130579,9.932573,531.959063,0.916619
9,electricity/15T/medium,moirai2_baseline,564.3730,9816.7,96.3,9720.4,13.112,7400,283337.976087,283337.976087,55.668058,0.841303,0.132270,0.130561,9.948002,532.295009,0.917198


In [11]:
def pivot(df, value_col, fmt=None):
    p = df.pivot(index="dataset", columns="config", values=value_col)
    p = p[[c for c in CONFIGS.keys() if c in p.columns]]  # preserve config order
    if fmt is not None:
        p = p.applymap(fmt)
    return p

print("\n=== Inference time (seconds) ===")
display(pivot(df, "inference_time_s", fmt=lambda x: f"{x:.2f}"))

print("\n=== Peak VRAM (MB) ===")
display(pivot(df, "peak_vram_mb", fmt=lambda x: f"{x:.0f}"))

print("\n=== Throughput (items/sec) ===")
display(pivot(df, "throughput_items_per_s", fmt=lambda x: f"{x:.2f}"))



=== Inference time (seconds) ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,0.21,0.91
bizitobs_l2c/H/medium,0.12,0.54
electricity/15T/long,179.19,938.88
electricity/15T/medium,88.33,564.37
ett1/H/long,0.54,2.64
ett1/H/medium,0.35,2.10
ett2/H/long,0.54,2.66
ett2/H/medium,0.35,2.11
jena_weather/H/long,1.04,5.27



=== Peak VRAM (MB) ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,292,682
bizitobs_l2c/H/medium,260,632
electricity/15T/long,3564,10690
electricity/15T/medium,3039,9817
ett1/H/long,667,1836
ett1/H/medium,745,2226
ett2/H/long,667,1836
ett2/H/medium,745,2226
jena_weather/H/long,1236,3576



=== Throughput (items/sec) ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,96.30,96.30
bizitobs_l2c/H/medium,96.30,96.30
electricity/15T/long,96.30,96.30
electricity/15T/medium,96.30,96.30
ett1/H/long,96.30,96.30
ett1/H/medium,96.30,96.30
ett2/H/long,96.30,96.30
ett2/H/medium,96.30,96.30
jena_weather/H/long,96.30,96.30


In [14]:
def speedup_vs(df, baseline_config):
    base = df[df["config"] == baseline_config].set_index("dataset")["inference_time_s"]
    out = df.copy()
    out["speedup_vs_" + baseline_config] = out.apply(
        lambda r: base[r["dataset"]] / r["inference_time_s"]
        if r["dataset"] in base.index and r["inference_time_s"] > 0 else np.nan,
        axis=1,
    )
    return out.pivot(index="dataset", columns="config",
                     values="speedup_vs_" + baseline_config)[
        [c for c in CONFIGS.keys() if c in out["config"].unique()]
    ]

# print("\n=== Speedup vs moiraic_original_no_cache (>1 = faster) ===")
# display(speedup_vs(df, "moiraic_original_no_cache").applymap(lambda x: f"{x:.2f}×" if pd.notna(x) else "—"))

print("\n=== Speedup vs moirai2_baseline (>1 = faster) ===")
display(speedup_vs(df, "moirai2_baseline").applymap(lambda x: f"{x:.2f}×" if pd.notna(x) else "—"))


=== Speedup vs moirai2_baseline (>1 = faster) ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,4.22×,1.00×
bizitobs_l2c/H/medium,4.64×,1.00×
electricity/15T/long,5.24×,1.00×
electricity/15T/medium,6.39×,1.00×
ett1/H/long,4.93×,1.00×
ett1/H/medium,6.07×,1.00×
ett2/H/long,4.94×,1.00×
ett2/H/medium,6.04×,1.00×
jena_weather/H/long,5.08×,1.00×


In [19]:
def VRAM_Usage_vs(df, baseline_config):
    base = df[df["config"] == baseline_config].set_index("dataset")["peak_vram_mb"]
    out = df.copy()
    out["VRAM_Usage_vs_" + baseline_config] = out.apply(
        lambda r: r["peak_vram_mb"] / base[r["dataset"]]
        if r["dataset"] in base.index and r["peak_vram_mb"] > 0 else np.nan,
        axis=1,
    )
    return out.pivot(index="dataset", columns="config",
                     values="VRAM_Usage_vs_" + baseline_config)[
        [c for c in CONFIGS.keys() if c in out["config"].unique()]
    ]

# print("\n=== VRAM_Usage vs moiraic_original_no_cache (>1 = faster) ===")
# display(VRAM_Usage_vs(df, "moiraic_original_no_cache").applymap(lambda x: f"{x:.2f}×" if pd.notna(x) else "—"))

print("\n=== VRAM_Usage vs moirai2_baseline (<1 = Less VRAM) ===")
display(VRAM_Usage_vs(df, "moirai2_baseline").applymap(lambda x: f"{x:.2f}×" if pd.notna(x) else "—"))


=== VRAM_Usage vs moirai2_baseline (<1 = Less VRAM) ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,0.43×,1.00×
bizitobs_l2c/H/medium,0.41×,1.00×
electricity/15T/long,0.33×,1.00×
electricity/15T/medium,0.31×,1.00×
ett1/H/long,0.36×,1.00×
ett1/H/medium,0.33×,1.00×
ett2/H/long,0.36×,1.00×
ett2/H/medium,0.33×,1.00×
jena_weather/H/long,0.35×,1.00×


In [15]:
# - `moiraic_cached` vs `moiraic_frozen_no_cache`: should be ~identical (same semantics, different
#   implementation). Any divergence is a cache correctness bug.
# - `moiraic_original_no_cache` vs `moirai2_baseline`: should be ~identical (default toggles
#   preserve original behavior). Sanity check that the new code is non-regressive.
# - `moiraic_cached` vs `moiraic_original_no_cache`: measures the forecast-quality cost of the
#   frozen-scaler approximation that the cache enforces.

# %%
for metric in KEEP_METRICS:
    print(f"\n=== {metric} ===")
    display(pivot(df, metric, fmt=lambda x: f"{x:.4f}"))

# %%
# Pairwise quality deltas vs cached path.
def delta_vs(df, baseline_config, metrics):
    base = df[df["config"] == baseline_config].set_index("dataset")
    rows = []
    for _, r in df.iterrows():
        if r["config"] == baseline_config or r["dataset"] not in base.index:
            continue
        for m in metrics:
            rows.append({
                "dataset": r["dataset"],
                "config": r["config"],
                "metric": m,
                "abs_delta": r[m] - base.loc[r["dataset"], m],
                "rel_delta_%": 100 * (r[m] - base.loc[r["dataset"], m])
                                / abs(base.loc[r["dataset"], m])
                                if base.loc[r["dataset"], m] != 0 else np.nan,
            })
    return pd.DataFrame(rows)

deltas = delta_vs(df, "moiraic_cached", KEEP_METRICS)

print("\n=== Quality delta vs moiraic_cached (positive = worse if metric is loss) ===")
for m in KEEP_METRICS:
    sub = deltas[deltas["metric"] == m].pivot(
        index="dataset", columns="config", values="rel_delta_%"
    )
    sub = sub[[c for c in CONFIGS.keys() if c in sub.columns]]
    print(f"\n{m} (relative delta %)")
    display(sub.applymap(lambda x: f"{x:+.2f}%" if pd.notna(x) else "—"))


=== MASE[0.5] ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,100.0280,99.5689
bizitobs_l2c/H/medium,73.5813,73.6055
electricity/15T/long,391279.4566,391376.2093
electricity/15T/medium,282980.4452,283337.9761
ett1/H/long,201.6448,203.1143
ett1/H/medium,166.7802,167.3300
ett2/H/long,236.8780,237.5621
ett2/H/medium,252.4579,253.3274
jena_weather/H/long,1278.7873,1278.8538



=== sMAPE[0.5] ===


config,moiraic_cached,moirai2_baseline
dataset,,
bizitobs_l2c/H/long,0.6118,0.6108
bizitobs_l2c/H/medium,0.5086,0.5086
electricity/15T/long,0.9077,0.9071
electricity/15T/medium,0.8412,0.8413
ett1/H/long,1.4652,1.4650
ett1/H/medium,1.3030,1.3043
ett2/H/long,1.0525,1.0515
ett2/H/medium,1.0847,1.0853
jena_weather/H/long,1.0387,1.0389



=== NRMSE[mean] ===


...

config,moirai2_baseline
dataset,
bizitobs_l2c/H/long,+0.24%
bizitobs_l2c/H/medium,+0.38%
electricity/15T/long,+0.23%
electricity/15T/medium,+0.16%
ett1/H/long,+0.38%
ett1/H/medium,+0.10%
ett2/H/long,+0.64%
ett2/H/medium,+0.31%
jena_weather/H/long,+0.94%



ND[0.5] (relative delta %)


config,moirai2_baseline
dataset,
bizitobs_l2c/H/long,-0.23%
bizitobs_l2c/H/medium,+0.02%
electricity/15T/long,+0.01%
electricity/15T/medium,+0.06%
ett1/H/long,+0.36%
ett1/H/medium,+0.16%
ett2/H/long,+0.14%
ett2/H/medium,+0.17%
jena_weather/H/long,+0.00%



mean_weighted_sum_quantile_loss (relative delta %)


config,moirai2_baseline
dataset,
bizitobs_l2c/H/long,-0.23%
bizitobs_l2c/H/medium,+0.02%
electricity/15T/long,+0.01%
electricity/15T/medium,+0.06%
ett1/H/long,+0.36%
ett1/H/medium,+0.16%
ett2/H/long,+0.14%
ett2/H/medium,+0.17%
jena_weather/H/long,+0.00%


In [16]:
agg_cols = ["inference_time_s", "peak_vram_mb", "throughput_items_per_s"] + KEEP_METRICS
summary = df.groupby("config")[agg_cols].mean().round(4)
summary = summary.loc[[c for c in CONFIGS.keys() if c in summary.index]]
print("=== Mean across datasets ===")
display(summary)

# Speedup/VRAM summary against each baseline.
for baseline in ["moiraic_original_no_cache", "moirai2_baseline"]:
    if baseline not in summary.index:
        continue
    s = pd.DataFrame({
        "mean_speedup": summary["throughput_items_per_s"] / summary.loc[baseline, "throughput_items_per_s"],
        "vram_ratio":   summary["peak_vram_mb"]            / summary.loc[baseline, "peak_vram_mb"],
    }).round(3)
    print(f"\n=== Aggregate vs {baseline} ===")
    display(s)

=== Mean across datasets ===


,inference_time_s,peak_vram_mb,throughput_items_per_s,MASE[0.5],sMAPE[0.5],NRMSE[mean],ND[0.5],mean_weighted_sum_quantile_loss
config,,,,,,,,
moiraic_cached,36.3833,1817.4071,92.5714,48439.9884,0.9592,13.0292,95.5205,0.5657
moirai2_baseline,205.8705,5558.9929,96.3000,48472.9231,0.9595,13.0797,95.5631,0.5662



=== Aggregate vs moirai2_baseline ===


,mean_speedup,vram_ratio
config,,
moiraic_cached,0.961,0.327
moirai2_baseline,1.000,1.000
